# MESA Validate

## Overview

Validate supports human review of LLM-extracted structured outputs against a Pydantic schema. A human reviewer works through a sample of documents, comparing the source text against the model's extraction, and marks each targeted schema element as correct or incorrect. From these judgements, Validate calculates precision, recall and F1.

>❗Validate normally runs as a Streamlit app. To illustrate its operation within a notebook context, its underlying logic is re-presented here via a set of widgets.

## Steps

1. Import the pieces of Validate (pre-added as a dependency to this repository) that we need: the data models for configuring a session, the classes that manage session state and inspect a schema, and the metrics helpers used at the end.

In [ ]:
from pathlib import Path

from notebook_ui import DocumentReviewer

from mesa_validate.models import FieldSelection, Session
from mesa_validate.session_manager import SessionManager
from mesa_validate.schema_inspector import SchemaInspector
from mesa_validate.metrics import (
    aggregate_metrics,
    export_to_csv_string,
    format_metrics_summary,
)

2. Pick a schema to validate against and inspect it to find the classes, fields and enum values available for selection.

> 📓 Validate already supports the oncology schema we have used throughout the notebooks, so we can pick that.

In [ ]:
inspector: SchemaInspector = SchemaInspector("oncoschema.schema", "OncologyModel")
sorted(inspector.classes)

3. Define what to validate. Each `FieldSelection` is one of:

   - `basemodel_field` — a single field on a class, e.g. `PrimaryCancerFacts.topography`
   - `basemodel_class` — an entire class, correct only if every field within it is correct
   - `enum_value` — one value of an enum used to tag items in a repeating class, e.g. a specific `TimelineEventType`

Below a set of example fields have already been chosen.

In [ ]:
selections: list[FieldSelection] = [
    FieldSelection(selection_type="basemodel_field", class_name="PrimaryCancerFacts", field_name="topography"),
    FieldSelection(selection_type="basemodel_field", class_name="PrimaryCancerFacts", field_name="morphology"),
    FieldSelection(selection_type="basemodel_field", class_name="PrimaryCancerFacts", field_name="diagnosis_year"),
    FieldSelection(selection_type="basemodel_field", class_name="PrimaryCancerFacts", field_name="diagnosis_month"),
    FieldSelection(selection_type="basemodel_field", class_name="PrimaryCancerFacts", field_name="tnm_stage"),
    FieldSelection(selection_type="basemodel_class", class_name="MolecularBiomarkerProfile"),
    FieldSelection(
        selection_type="enum_value",
        class_name="BiomarkerStatus",
        enum_value="negative",
    ),
    FieldSelection(selection_type="basemodel_class", class_name="PerformanceStatus"),
    FieldSelection(
        selection_type="enum_value",
        class_name="TimelineEventType",
        enum_value="experienced_toxicity_or_complication_related_to_treatment",
    ),
    FieldSelection(
        selection_type="enum_value",
        class_name="TimelineEventType",
        enum_value="evidence_of_metastatic_progression",
    ),
    FieldSelection(
        selection_type="enum_value",
        class_name="TimelineEventType",
        enum_value="radiology_evidence_of_disease_progression",
    ),
    FieldSelection(
        selection_type="enum_value",
        class_name="TimelineEventType",
        enum_value="experienced_treatment_reduction_or_stop",
    ),
]

4. Create a session. A session pins together a schema, a folder of prediction files to sample from, and the selections above. Point `predictions_folder` at your own model output (e.g. the Deploy and Runner notebook's `../deploy_and_runner/data/output`); here we use its bundled sample data, `../data/sample`, so this notebook runs standalone. `sample_size` caps how many documents are drawn, and documents that fail schema validation are excluded automatically and reported separately, not counted towards the sample.

>❗The session `id` below is fixed rather than randomly generated, so re-running this cell resumes the same session (and its saved judgements) instead of starting a fresh one. Change it, or delete `sessions/notebook-demo`, to start over.

In [ ]:
session: Session = Session(
    id="notebook-demo",
    name="Notebook demo",
    schema_module="oncoschema.schema",
    root_class="OncologyModel",
    predictions_folder="../data/sample",
    sample_size=5,
    selections=selections,
)

manager: SessionManager = SessionManager.create_new(session)
progress, excluded = manager.initialize_files(session)

5. Review each document. Choose `None` / `Correct` / `Incorrect` for each item; list-style selections also take a count of items present in the text but missed by the model. Tick **Mark document as fully validated** once you're done with a document.

In [ ]:
reviewer: DocumentReviewer = DocumentReviewer(session, inspector, manager, progress, excluded)
reviewer

>❗Re-running the cell above resets the widget to the first document but keeps all saved judgements (they live in `sessions/notebook-demo/progress.json`, not in the widget). If you restart the kernel, just re-run every cell from step 4 onwards.

> 📓 It is during the evaluation process that we would recognise the performance of our model on the KRAS problem.

6. Once a useful number of documents are marked as fully validated, calculate precision, recall and F1 per selection.

In [ ]:
def render_table(rows):
    columns = list(rows[0].keys())
    print(" | ".join(columns))
    for row in rows:
        print(" | ".join(str(row[column]) for column in columns))
    print()

progress = manager.load_progress()

if not progress.get("completed_files"):
    print("No completed documents yet.")
else:
    summary = format_metrics_summary(aggregate_metrics(progress, session.selections))
    render_table([row for row in summary if row["type"] == "Binary"])
    render_table([row for row in summary if row["type"] == "List"])

7. Export the same summary as CSV.

In [ ]:
csv_data = export_to_csv_string(progress, session.selections)
Path("results.csv").write_text(csv_data)
print(csv_data)